## Data Normalization

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

### CLeaning
1. Accent stripping

In [6]:
def strip_accents(text):
    if not isinstance(text, str):
        return ''
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

strip_accents('Caf\u00e9 M\u00fcller \u2014 fa\u00e7ade \u2116 12'), strip_accents('stra\u00dfe')

('Cafe Muller  facade No 12', 'strae')

2. Name normalization

In [9]:
NAME_ABBREV = {
    'incorporated': 'inc', 'inc': 'inc',
    'corporation': 'corp', 'corp': 'corp',
    'limited': 'ltd', 'ltd': 'ltd',
    'company': 'co', 'co': 'co',
    'private': 'pvt', 'pvt': 'pvt',
    'international': 'intl', 'intl': 'intl',
    'enterprises': 'ent', 'ent': 'ent',
    'services': 'svc', 'svc': 'svc',
}

LEGAL_SUFFIXES = {'inc', 'corp', 'ltd', 'llc', 'co', 'pvt', 'ent', 'svc',
                  'industries', 'ind', 'group', 'grp', 'holdings', 'hldg'}

def clean_name(name, drop_legal_suffix=False):
    if not isinstance(name, str) or not name.strip():
        return 
    t = strip_accents(name).lower()
    t = t.replace('&', ' and ')
    t = re.sub(r"[^a-z0-9\\s]", ' ', t) 
    tokens = [NAME_ABBREV.get(w, w) for w in t.split()]
    if drop_legal_suffix:
        tokens = [w for w in tokens if w not in LEGAL_SUFFIXES]
    return ' '.join(tokens)

  

Testing!

In [10]:
for raw in ["MCDONALD'S CORPORATION", "McDonald's Corp.",
            'Caf\u00e9 & Bar Pvt. Ltd.', 'Tata Consultancy Services Ltd',
            'STARBUCKS & CO', 'starbucks company']:
    print(f'{raw!r:40} -> {clean_name(raw)!r:32} | blocking: {clean_name(raw, True)!r}')

"MCDONALD'S CORPORATION"                 -> 'mcdonald s corp'                | blocking: 'mcdonald s'
"McDonald's Corp."                       -> 'mcdonald s corp'                | blocking: 'mcdonald s'
'Café & Bar Pvt. Ltd.'                   -> 'cafe and bar pvt ltd'           | blocking: 'cafe and bar'
'Tata Consultancy Services Ltd'          -> 'tata consultancy svc ltd'       | blocking: 'tata consultancy'
'STARBUCKS & CO'                         -> 'starbucks and co'               | blocking: 'starbucks and'
'starbucks company'                      -> 'starbucks co'                   | blocking: 'starbucks'
